# Optional Notebook 5C — Downloading EEG and visualizing ICA with MNE-Python

**Quantitative Methods in Neuroscience · Master in Neuroscience · University of Geneva**  
**Optional extension — not assessed**

This notebook is a short demonstration of how a neuroscience package can:

1. download a public EEG recording;
2. represent continuous EEG as an `mne.io.Raw` object;
3. inspect the signal and its power spectrum;
4. prepare the data for independent component analysis (ICA);
5. plot ICA component topographies and time courses;
6. apply artifact removal only after inspecting the components.

The example uses **one approximately 61-second recording** from subject 1 of the EEG Motor Movement/Imagery Dataset distributed through PhysioNet and accessed with MNE's dataset fetcher.

> The goal is to see the workflow and software objects. This is not a complete EEG preprocessing protocol.

## 0 · Before running

Use the dedicated **Python (qmn-optional)** environment. From this folder, create it once:

```bash
conda env create -f environment.yml
conda activate qmn-optional
python -m ipykernel install --user --name qmn-optional --display-name "Python (qmn-optional)"
jupyter lab
```

The first execution downloads one EDF file into `notebooks/data/optional_mne/`. Later executions reuse the local copy.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import mne
from mne.preprocessing import ICA

mne.set_log_level("WARNING")

# Keep True for the student demonstration. It is a switch only for offline testing.
RUN_DOWNLOAD = True

DATA_DIR = Path("data") / "optional_mne"
SUBJECT = 1
RUNS = [1]  # one-minute eyes-open baseline recording

print("Python:", sys.version.split()[0])
print("MNE-Python:", mne.__version__)
print("Download/cache directory:", DATA_DIR.resolve())

## 1 · Download one public EEG file

`mne.datasets.eegbci.load_data()` is a **dataset fetcher**. It downloads the requested file only when it is absent from the cache and returns the local path.

Downloading only one subject and one run keeps the optional demonstration small.

In [ ]:
if RUN_DOWNLOAD:
    eeg_paths = mne.datasets.eegbci.load_data(
        subjects=SUBJECT,
        runs=RUNS,
        path=DATA_DIR,
    )
    eeg_path = Path(eeg_paths[0])
    print("EEG file:", eeg_path)
    print(f"File size: {eeg_path.stat().st_size / 1024**2:.1f} MB")
else:
    eeg_path = None
    print("Download skipped in offline-validation mode.")

## 2 · Read the EDF recording and attach sensor positions

MNE stores continuous electrophysiology in a `Raw` object. The object combines:

- the signal array;
- channel names and channel types;
- sampling frequency;
- annotations;
- sensor positions and other metadata.

In [ ]:
if RUN_DOWNLOAD:
    raw = mne.io.read_raw_edf(eeg_path, preload=True, verbose=False)

    # Convert the EEGBCI channel labels into standard names and attach a montage.
    mne.datasets.eegbci.standardize(raw)
    montage = mne.channels.make_standard_montage("standard_1005")
    raw.set_montage(montage, on_missing="warn")

    summary = {
        "duration_s": raw.times[-1],
        "sampling_frequency_hz": raw.info["sfreq"],
        "n_channels": raw.info["nchan"],
        "channel_types": sorted(set(raw.get_channel_types())),
    }
    display(summary)
else:
    print("Raw object will be created when RUN_DOWNLOAD is True.")

## 3 · Plot a short EEG segment

A raw trace should be inspected before filtering or artifact correction. Look for:

- channels with unusually large amplitudes;
- slow drifts;
- brief high-frequency bursts;
- signals shared by many frontal channels, which may reflect eye activity.

In [ ]:
if RUN_DOWNLOAD:
    raw_preview = raw.copy().pick("eeg").crop(tmin=0, tmax=12)
    fig = raw_preview.plot(
        duration=12,
        n_channels=20,
        scalings="auto",
        show_scrollbars=False,
        show=False,
        title="EEGBCI subject 1, run 1 — first 12 seconds",
    )
    plt.show()

## 4 · Inspect the spectrum

The power spectral density summarizes how signal power is distributed across frequency. It is useful for checking line noise, slow drift, and unusually noisy channels.

In [ ]:
if RUN_DOWNLOAD:
    spectrum = raw.copy().pick("eeg").compute_psd(fmin=1, fmax=45, verbose=False)
    spectrum.plot(average=True, spatial_colors=False, show=False)
    plt.show()

## 5 · Prepare a separate copy for ICA

ICA is sensitive to low-frequency drift. MNE's documentation recommends high-pass filtering before fitting ICA, commonly at approximately 1 Hz.

Conceptually, ICA searches for latent source signals whose mixtures reconstruct the observed channels:

\[
\mathbf{X} \approx \mathbf{A}\mathbf{S},
\]

where \(\mathbf{X}\) is the channel-by-time EEG matrix, \(\mathbf{S}\) contains independent component time courses, and \(\mathbf{A}\) contains their sensor projections.

The ICA-filtered copy is used to **estimate the decomposition**. A carefully selected set of components can later be removed from another copy of the data.

In [ ]:
if RUN_DOWNLOAD:
    raw_for_ica = raw.copy().pick("eeg")
    raw_for_ica.filter(l_freq=1.0, h_freq=40.0, verbose=False)

    # Average reference is common for EEG. Applying it directly keeps the example simple.
    raw_for_ica.set_eeg_reference("average", projection=False, verbose=False)

    print(raw_for_ica)

## 6 · Fit ICA

For a short classroom demonstration, we estimate 15 components and decimate the fitting data. A real analysis should justify the number of components, evaluate data rank, inspect bad channels, and document every exclusion.

In [ ]:
if RUN_DOWNLOAD:
    ica = ICA(
        n_components=15,
        method="fastica",
        random_state=97,
        max_iter="auto",
    )
    ica.fit(raw_for_ica, decim=2, verbose=False)
    print(ica)

## 7 · Plot ICA component topographies

Each topography shows how one independent component projects onto the EEG sensors. Artifact components are identified from **multiple pieces of evidence**, not from topography alone.

In [ ]:
if RUN_DOWNLOAD:
    ica.plot_components(
        picks=range(15),
        inst=raw_for_ica,
        show_names=True,
        show=False,
    )
    plt.show()

## 8 · Plot the component time courses

The component sources make transient bursts, slow movements, and periodic artifacts easier to recognize.

In [ ]:
if RUN_DOWNLOAD:
    ica.plot_sources(
        raw_for_ica,
        start=0,
        stop=20,
        show=False,
        title="ICA source time courses — inspect before excluding",
    )
    plt.show()

## 9 · Optional automated muscle-artifact scores

MNE can compute heuristic muscle-artifact scores. These scores are a **screening aid**, not a substitute for visual inspection or experimental knowledge.

In [ ]:
if RUN_DOWNLOAD:
    muscle_components, muscle_scores = ica.find_bads_muscle(raw_for_ica)
    print("Components flagged by the muscle heuristic:", muscle_components)

    ica.plot_scores(
        muscle_scores,
        exclude=muscle_components,
        title="Muscle-artifact heuristic scores",
        show=False,
    )
    plt.show()

## 10 · Demonstrate component removal safely

Enter component indices only after inspecting their topographies, time courses, spectra, and relationship with known artifact channels.

The default list is empty, so the notebook does **not** remove anything automatically.

In [ ]:
if RUN_DOWNLOAD:
    COMPONENTS_TO_REMOVE = []  # Example after inspection: [0]

    raw_clean = raw.copy().pick("eeg").load_data()
    ica.exclude = COMPONENTS_TO_REMOVE
    ica.apply(raw_clean, verbose=False)

    print("Removed components:", COMPONENTS_TO_REMOVE)
    print("A cleaned copy now exists as raw_clean; the original raw object was preserved.")

## What students should remember

- Dataset fetchers make public data accessible with reproducible code.
- `Raw` combines signals and metadata.
- Always inspect the raw signal before preprocessing.
- Fit ICA on data with slow drift controlled.
- ICA components are **not automatically brain or artifact**.
- Preserve the original data and document every excluded component.

## References and documentation

- MNE-Python documentation: [Datasets](https://mne.tools/stable/api/datasets.html), [Repairing artifacts with ICA](https://mne.tools/stable/auto_tutorials/preprocessing/40_artifact_correction_ica.html), and [`mne.preprocessing.ICA`](https://mne.tools/stable/generated/mne.preprocessing.ICA.html).
- Schalk, G. et al. (2004). **“BCI2000: A General-Purpose Brain-Computer Interface (BCI) System.”** *IEEE Transactions on Biomedical Engineering*, 51(6), 1034–1043.